In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [3]:
books = pd.read_csv("../data/books.csv")
ratings = pd.read_csv("../data/ratings.csv")
book_tags = pd.read_csv("../data/book_tags.csv")
tags = pd.read_csv("../data/tags.csv")

In [4]:
books_clean = books[
    [
        "book_id",
        "title",
        "authors",
        "original_publication_year",
        "average_rating",
        "ratings_count",
        "image_url"
    ]
].copy()

In [5]:
books_clean.head()

,book_id,title,authors,original_publication_year,average_rating,ratings_count,image_url
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,2008.0,4.34,4780653,https://images.gr-assets.com/books/1447303603m...
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",1997.0,4.44,4602479,https://images.gr-assets.com/books/1474154022m...
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,2005.0,3.57,3866839,https://images.gr-assets.com/books/1361039443m...
3,4,To Kill a Mockingbird,Harper Lee,1960.0,4.25,3198671,https://images.gr-assets.com/books/1361975680m...
4,5,The Great Gatsby,F. Scott Fitzgerald,1925.0,3.89,2683664,https://images.gr-assets.com/books/1490528560m...


In [6]:
books_clean.shape

(10000, 7)

In [7]:
books_clean[books_clean["original_publication_year"].isna()][
    ["book_id", "title", "original_publication_year"]
]

,book_id,title,original_publication_year
219,220,Twilight: The Complete Illustrated Movie Compa...,NaN
975,976,"Dr. Seuss's Green Eggs and Ham: For Soprano, B...",NaN
3505,3506,"Absolute Boyfriend, Vol. 1",NaN
4228,4229,The Lord of the Rings: Weapons and Warfare,NaN
4247,4248,Marked / Betrayed / Chosen / Untamed / Hunted ...,NaN
4409,4410,The Lord of the Rings: The Return of the King:...,NaN
4707,4708,BookRags Summary: A Storm of Swords,NaN
4770,4771,"The Doomsday Key (Sigma Force, #6)",NaN
4877,4878,"A Shade of Blood (A Shade of Vampire, #2)",NaN
5609,5610,The 8th Habit: From Effectiveness to Greatness,NaN


In [8]:
ratings.duplicated().sum()

np.int64(0)

In [9]:
ratings.duplicated(
    subset=["user_id", "book_id"]
).sum()

np.int64(0)

In [10]:
user_item_matrix = ratings.pivot(
    index="user_id",
    columns="book_id",
    values="rating"
)

In [11]:
user_item_matrix.shape

(53424, 10000)

In [12]:
book_id_map = books[
    ["book_id", "goodreads_book_id"]
].copy()

In [13]:
book_tags = book_tags.merge(
    book_id_map,
    on="goodreads_book_id",
    how="inner"
)

In [14]:
book_tags.head()

,goodreads_book_id,tag_id,count,book_id
0,1,30574,167697,27
1,1,11305,37174,27
2,1,11557,34173,27
3,1,8717,12986,27
4,1,33114,12716,27


In [15]:
ratings.duplicated().sum()

np.int64(0)

In [16]:
ratings.duplicated(
    subset=["user_id", "book_id"]
).sum()

np.int64(0)

In [17]:
user_item_matrix.shape

(53424, 10000)

In [18]:
book_tags.shape

(999912, 4)

In [19]:
book_tags.shape

(999912, 4)

In [20]:
tag_usage = (
    book_tags
    .groupby("tag_id")["count"]
    .sum()
    .sort_values(ascending=False)
)

In [21]:
tag_usage.describe()

count    3.425200e+04
mean     6.097491e+03
std      7.627092e+05
min     -1.000000e+00
25%      3.000000e+00
50%      1.000000e+01
75%      5.200000e+01
max      1.407188e+08
Name: count, dtype: float64

In [22]:
tag_usage.head(20)

tag_id
30574    140718761
8717       7507958
11557      4503173
11743      3688819
11305      3548157
33114      1848306
7457       1756920
5207       1317235
26138      1231926
22743      1224279
32989       898334
20939       872282
21689       857901
14487       815421
27199       782637
26837       703866
26771       597325
22973       542559
17213       506882
8055        486001
Name: count, dtype: int64

In [23]:
tag_usage_df = (
    tag_usage
    .reset_index()
    .merge(tags, on="tag_id", how="left")
)

In [24]:
tag_usage_df.head(30)

,tag_id,count,tag_name
0,30574,140718761,to-read
1,8717,7507958,currently-reading
2,11557,4503173,favorites
3,11743,3688819,fiction
4,11305,3548157,fantasy
5,33114,1848306,young-adult
6,7457,1756920,classics
7,5207,1317235,books-i-own
8,26138,1231926,romance
9,22743,1224279,owned


In [25]:
tag_usage = (
    book_tags
    .groupby("tag_id")["count"]
    .sum()
    .sort_values(ascending=False)
)

In [26]:
tag_usage.describe()

count    3.425200e+04
mean     6.097491e+03
std      7.627092e+05
min     -1.000000e+00
25%      3.000000e+00
50%      1.000000e+01
75%      5.200000e+01
max      1.407188e+08
Name: count, dtype: float64

In [27]:
tag_usage_df = (
    tag_usage
    .reset_index()
    .merge(tags, on="tag_id", how="left")
)

In [28]:
tag_usage_df.head(30)

,tag_id,count,tag_name
0,30574,140718761,to-read
1,8717,7507958,currently-reading
2,11557,4503173,favorites
3,11743,3688819,fiction
4,11305,3548157,fantasy
5,33114,1848306,young-adult
6,7457,1756920,classics
7,5207,1317235,books-i-own
8,26138,1231926,romance
9,22743,1224279,owned


In [29]:
tag_book_frequency = (
    book_tags
    .groupby("tag_id")["book_id"]
    .nunique()
    .sort_values(ascending=False)
)

In [30]:
tag_book_frequency.describe()

count    34252.000000
mean        29.192573
std        277.251721
min          1.000000
25%          1.000000
50%          1.000000
75%          5.000000
max       9983.000000
Name: book_id, dtype: float64

In [31]:
tag_frequency_df = (
    tag_book_frequency
    .reset_index(name="book_frequency")
    .merge(tags, on="tag_id", how="left")
    .sort_values("book_frequency", ascending=False)
)

In [32]:
tag_frequency_df.head(30)

,tag_id,book_frequency,tag_name
0,30574,9983,to-read
1,11557,9881,favorites
2,22743,9857,owned
3,5207,9799,books-i-own
4,8717,9776,currently-reading
5,18045,9415,library
6,22753,9221,owned-books
7,11743,9097,fiction
8,30521,8692,to-buy
9,17213,8316,kindle


In [33]:
tag_frequency_df[
    tag_frequency_df["book_frequency"] >= 100
].shape

(1109, 3)

In [34]:
tag_frequency_df[tag_frequency_df["book_frequency"] >= 10].shape

(5503, 3)

In [35]:
tag_frequency_df[tag_frequency_df["book_frequency"] >= 20].shape

(3359, 3)

In [36]:
tag_frequency_df[tag_frequency_df["book_frequency"] >= 50].shape

(1802, 3)

In [37]:
content_tags = tag_frequency_df[
    tag_frequency_df["book_frequency"] >= 10
].copy()

In [38]:
content_tags["tag_name"].head(50).tolist()

['to-read',
 'favorites',
 'owned',
 'books-i-own',
 'currently-reading',
 'library',
 'owned-books',
 'fiction',
 'to-buy',
 'kindle',
 'default',
 'ebook',
 'my-books',
 'audiobook',
 'ebooks',
 'wish-list',
 'my-library',
 'audiobooks',
 'i-own',
 'adult',
 'audio',
 'favourites',
 'novels',
 'own-it',
 'contemporary',
 'read-in-2015',
 'series',
 'e-book',
 'read-in-2016',
 'read-in-2014',
 'books',
 'adult-fiction',
 'e-books',
 'read-in-2013',
 'book-club',
 'audible',
 'fantasy',
 'romance',
 'audio-books',
 'abandoned',
 'novel',
 're-read',
 'have',
 'audio-book',
 'mystery',
 'borrowed',
 'adventure',
 'read-in-2012',
 'young-adult',
 'english']

In [39]:
content_tags["tag_name"].tail(50).tolist()

['selling',
 'self-help-spiritual',
 'truecrime',
 'health-and-nutrition',
 'danish-literature',
 'true-life',
 'burroughs',
 'harry-hole',
 'harry-hole-series',
 'fyic',
 'books-to-read-in-your-20-s',
 'also-a-movie',
 'المفضلة',
 'fated-to-be-mated',
 'heinlein-robert',
 'heaven',
 'lm-montgomery',
 'lords-of-the-underworld',
 'رواياتي',
 'conspiracy-thriller',
 'conservation',
 'indonesia-books',
 'roman-history',
 'amazon-freebies',
 'connecticut',
 'رعب',
 'intermediate',
 'international-development',
 'stockholm-syndrome',
 'steve-berry',
 'sci-fiction',
 'sophia',
 'sons',
 'software',
 'the-malazan-book-of-the-fallen',
 'the-legend-of-drizzt-series',
 'gilmore-reading-challenge',
 'crais-robert',
 'mybooks',
 'homeschooling',
 'homelessness',
 'batman-collection',
 'ciência',
 'mundodisco',
 'multi-cultural',
 'sports-non-fiction',
 'author-gena-showalter',
 'nyrb',
 'nutrition-health',
 'french-history']

In [40]:
excluded_tags = {
    "to-read",
    "currently-reading",
    "favorites",
    "favourites",
    "owned",
    "books-i-own",
    "owned-books",
    "to-buy",
    "library",
    "kindle",
    "default",
    "ebook",
    "ebooks",
    "my-books",
    "my-library",
    "i-own",
    "own-it",
    "wish-list",
    "audiobook",
    "audiobooks",
    "audio",
    "borrowed"
}

In [41]:
content_tags = tag_frequency_df[
    tag_frequency_df["book_frequency"] >= 10
].copy()

In [42]:
content_tags = content_tags[
    ~content_tags["tag_name"].isin(excluded_tags)
]

In [43]:
content_tags = content_tags[
    ~content_tags["tag_name"].str.match(
        r"^(read-in|to-read-in)-\d{4}$",
        na=False
    )
]

In [44]:
content_tags.shape

(5468, 3)

In [45]:
content_tags["tag_name"].head(50).tolist()

['fiction',
 'adult',
 'novels',
 'contemporary',
 'series',
 'e-book',
 'books',
 'adult-fiction',
 'e-books',
 'book-club',
 'audible',
 'fantasy',
 'romance',
 'audio-books',
 'abandoned',
 'novel',
 're-read',
 'have',
 'audio-book',
 'mystery',
 'adventure',
 'young-adult',
 'english',
 'did-not-finish',
 'favorite',
 'maybe',
 'shelfari-favorites',
 'drama',
 'literature',
 'general-fiction',
 'read-2015',
 'ya',
 'all-time-favorites',
 'classics',
 'read-2016',
 'read-2014',
 'contemporary-fiction',
 'favorite-books',
 'dnf',
 'finished',
 '5-stars',
 'historical-fiction',
 'paperback',
 'historical',
 'thriller',
 'sci-fi-fantasy',
 'american',
 'suspense',
 'reviewed',
 '4-stars']

In [46]:
excluded_tags.update({
    "abandoned",
    "re-read",
    "have",
    "did-not-finish",
    "favorite",
    "reviewed",
    "4-stars"
})

In [47]:
content_tags = tag_frequency_df[
    tag_frequency_df["book_frequency"] >= 10
].copy()

content_tags = content_tags[
    ~content_tags["tag_name"].isin(excluded_tags)
]

content_tags = content_tags[
    ~content_tags["tag_name"].str.match(
        r"^(read-in|to-read-in)-\d{4}$",
        na=False
    )
]

In [48]:
content_tags.shape

(5461, 3)

In [49]:
book_tags_content = book_tags.merge(
    content_tags[["tag_id", "tag_name"]],
    on="tag_id",
    how="inner"
)

In [50]:
book_tags_content.shape

(710585, 5)

In [51]:
book_tags_content.head()

,goodreads_book_id,tag_id,count,book_id,tag_name
0,1,11305,37174,27,fantasy
1,1,33114,12716,27,young-adult
2,1,11743,9954,27,fiction
3,1,14017,7169,27,harry-potter
4,1,32989,4364,27,ya


In [52]:
book_tag_text = (
    book_tags_content
    .groupby("book_id")["tag_name"]
    .apply(" ".join)
    .reset_index()
)

In [53]:
book_tag_text.head()

,book_id,tag_name
0,1,young-adult fiction dystopian dystopia fantasy...
1,2,fantasy young-adult fiction harry-potter ya se...
2,3,young-adult fantasy vampires ya fiction parano...
3,4,classics classic historical-fiction school clà...
4,5,classics fiction classic literature school his...


In [54]:
book_tag_text.shape

(10000, 2)

In [55]:
books_features = books_clean.merge(
    book_tag_text,
    on="book_id",
    how="left"
)

In [56]:
books_features.shape

(10000, 8)

In [57]:
books_features[
    ["book_id", "title", "authors", "tag_name"]
].head()

,book_id,title,authors,tag_name
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,young-adult fiction dystopian dystopia fantasy...
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",fantasy young-adult fiction harry-potter ya se...
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,young-adult fantasy vampires ya fiction parano...
3,4,To Kill a Mockingbird,Harper Lee,classics classic historical-fiction school clà...
4,5,The Great Gatsby,F. Scott Fitzgerald,classics fiction classic literature school his...


In [58]:
tfidf = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[\w-]+\b"
)

In [59]:
tfidf_matrix = tfidf.fit_transform(
    books_features["tag_name"].fillna("")
)

In [60]:
tfidf_matrix.shape

(10000, 5460)

In [61]:
feature_names = tfidf.get_feature_names_out()

In [62]:
len(feature_names)

5460

In [63]:
hunger_games_idx = books_features[
    books_features["title"].str.contains(
        "The Hunger Games",
        case=False,
        na=False
    )
].index[0]

In [64]:
hunger_games_vector = tfidf_matrix[hunger_games_idx]

In [65]:
hunger_games_vector.nnz

70

In [66]:
hunger_games_scores = hunger_games_vector.toarray().flatten()

top_indices = hunger_games_scores.argsort()[-15:][::-1]

list(
    zip(
        feature_names[top_indices],
        hunger_games_scores[top_indices]
    )
)

[('suzanne-collins', np.float64(0.2682289160474678)),
 ('distopia', np.float64(0.19362477677561368)),
 ('distopian', np.float64(0.1903885579803173)),
 ('ya-dystopian', np.float64(0.1859048616836805)),
 ('dystopias', np.float64(0.18083527479502226)),
 ('dystopian-fiction', np.float64(0.18040844223821995)),
 ('favourite-books', np.float64(0.17129370484298442)),
 ('future', np.float64(0.16299701212979742)),
 ('futuristic', np.float64(0.15953800593153347)),
 ('completed-series', np.float64(0.15270820064885032)),
 ('post-apocalyptic', np.float64(0.1507264319614445)),
 ('finished-series', np.float64(0.14893682554940058)),
 ('trilogy', np.float64(0.14833122663731477)),
 ('love-triangle', np.float64(0.1474836581335337)),
 ('survival', np.float64(0.1441299840455627))]

In [67]:
similarity_scores = cosine_similarity(
    hunger_games_vector,
    tfidf_matrix
).flatten()

In [68]:
similar_book_indices = similarity_scores.argsort()[-11:][::-1]

In [69]:
recommendations = books_features.iloc[similar_book_indices][
    ["book_id", "title", "authors"]
].copy()

In [70]:
recommendations = recommendations[
    recommendations["book_id"] != books_features.iloc[hunger_games_idx]["book_id"]
]

In [71]:
recommendations.head(10)

,book_id,title,authors
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins
506,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins
11,12,"Divergent (Divergent, #1)",Veronica Roth
90,91,"The Maze Runner (Maze Runner, #1)",James Dashner
326,327,"Legend (Legend, #1)",Marie Lu
68,69,"Insurgent (Divergent, #2)",Veronica Roth
2300,2301,"The Ask and the Answer (Chaos Walking, #2)",Patrick Ness
524,525,"Pandemonium (Delirium, #2)",Lauren Oliver
654,655,"Crossed (Matched, #2)",Ally Condie


In [72]:
def recommend_books(title, n=10):
    
    # Find the book
    matches = books_features[
        books_features["title"].str.contains(
            title,
            case=False,
            na=False
        )
    ]
    
    if matches.empty:
        return "Book not found."
    
    book_idx = matches.index[0]
    
    # Calculate similarity
    scores = cosine_similarity(
        tfidf_matrix[book_idx],
        tfidf_matrix
    ).flatten()
    
    # Get highest scores
    similar_indices = scores.argsort()[-(n + 1):][::-1]
    
    # Remove the input book itself
    similar_indices = [
        i for i in similar_indices
        if i != book_idx
    ][:n]
    
    recommendations = books_features.iloc[similar_indices][
        ["book_id", "title", "authors"]
    ].copy()
    
    recommendations["similarity"] = scores[similar_indices]
    
    return recommendations

In [73]:
recommend_books("The Hunger Games")

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.890083
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.862205
506,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.763699
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.700538
90,91,"The Maze Runner (Maze Runner, #1)",James Dashner,0.617344
326,327,"Legend (Legend, #1)",Marie Lu,0.595743
68,69,"Insurgent (Divergent, #2)",Veronica Roth,0.589065
2300,2301,"The Ask and the Answer (Chaos Walking, #2)",Patrick Ness,0.578314
524,525,"Pandemonium (Delirium, #2)",Lauren Oliver,0.563836
654,655,"Crossed (Matched, #2)",Ally Condie,0.560962


In [74]:
recommend_books("Harry Potter")

,book_id,title,authors,similarity
22,23,Harry Potter and the Chamber of Secrets (Harry...,"J.K. Rowling, Mary GrandPré",0.943239
17,18,Harry Potter and the Prisoner of Azkaban (Harr...,"J.K. Rowling, Mary GrandPré, Rufus Beck",0.937284
24,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.894873
26,27,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré",0.886347
23,24,Harry Potter and the Goblet of Fire (Harry Pot...,"J.K. Rowling, Mary GrandPré",0.883943
398,399,The Tales of Beedle the Bard,J.K. Rowling,0.613436
421,422,"Harry Potter Boxset (Harry Potter, #1-7)",J.K. Rowling,0.581982
1285,1286,Quidditch Through the Ages,"Kennilworthy Whisp, J.K. Rowling",0.581230
3752,3753,"Harry Potter Collection (Harry Potter, #1-6)",J.K. Rowling,0.580416
6140,6141,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling,0.561822


In [75]:
recommend_books("The Great Gatsby")

,book_id,title,authors,similarity
7,8,The Catcher in the Rye,J.D. Salinger,0.834309
3,4,To Kill a Mockingbird,Harper Lee,0.747734
31,32,Of Mice and Men,John Steinbeck,0.701787
27,28,Lord of the Flies,William Golding,0.645693
57,58,The Adventures of Huckleberry Finn,"Mark Twain, John Seelye, Guy Cardwell",0.611647
13,14,Animal Farm,George Orwell,0.592691
1565,1566,Ethan Frome,Edith Wharton,0.563185
137,138,The Scarlet Letter,"Nathaniel Hawthorne, Thomas E. Connolly, Nina ...",0.551018
130,131,The Grapes of Wrath,John Steinbeck,0.539381
129,130,The Old Man and the Sea,Ernest Hemingway,0.535099


In [76]:
user_item_matrix = ratings.pivot(
    index="user_id",
    columns="book_id",
    values="rating"
)

In [77]:
user_item_matrix.shape

(53424, 10000)

In [78]:
id="2m5y3d"
total_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]

rated_cells = user_item_matrix.notna().sum().sum()

sparsity = 1 - (rated_cells / total_cells)

sparsity

np.float64(0.988813119571728)

In [79]:
id="7r1q8s"
rated_cells

np.int64(5976479)

In [80]:
user_item_sparse = csr_matrix(
    user_item_matrix.fillna(0).values
)

In [81]:
user_item_sparse.shape

(53424, 10000)

In [82]:
user_item_sparse.nnz

5976479

In [83]:
item_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

In [84]:
item_model.fit(user_item_sparse.T)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [85]:
hunger_games_idx

np.int64(0)

In [86]:
distances, indices = item_model.kneighbors(
    user_item_sparse.T[hunger_games_idx],
    n_neighbors=11
)

In [87]:
distances = distances.flatten()
indices = indices.flatten()

In [88]:
collab_recommendations = books_features.iloc[indices][
    ["book_id", "title", "authors"]
].copy()

collab_recommendations["similarity"] = 1 - distances

collab_recommendations

,book_id,title,authors,similarity
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,1.000000
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.723011
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.687347
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",0.589840
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,0.561142
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.523317
30,31,The Help,Kathryn Stockett,0.459008
15,16,"The Girl with the Dragon Tattoo (Millennium, #1)","Stieg Larsson, Reg Keeland",0.458558
20,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.450046
3,4,To Kill a Mockingbird,Harper Lee,0.448696


In [89]:
collab_recommendations = collab_recommendations[
    collab_recommendations["book_id"] != 
    books_features.iloc[hunger_games_idx]["book_id"]
].head(10)

In [90]:
collab_recommendations

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.723011
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.687347
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",0.589840
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,0.561142
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.523317
30,31,The Help,Kathryn Stockett,0.459008
15,16,"The Girl with the Dragon Tattoo (Millennium, #1)","Stieg Larsson, Reg Keeland",0.458558
20,21,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",0.450046
3,4,To Kill a Mockingbird,Harper Lee,0.448696
24,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.448337


In [92]:
title_to_index = pd.Series(
    books_features.index,
    index=books_features["title"]
).drop_duplicates()

In [101]:
def recommend_content(title, n=10):

    # Find the index of the selected book
    idx = title_to_index[title]

    # Calculate similarity of this book with every other book
    similarities = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    # Get indices of books sorted by similarity
    similar_indices = similarities.argsort()[::-1]

    # Remove the selected book itself
    similar_indices = [
        i for i in similar_indices
        if i != idx
    ]

    # Take top n
    top_indices = similar_indices[:n]

    # Create result
    result = books_features.iloc[
        top_indices
    ][["book_id", "title", "authors"]].copy()

    # Add similarity scores
    result["similarity"] = similarities[top_indices]

    return result.reset_index(drop=True)

In [95]:
books_features[
    books_features["title"].str.contains(
        "Hunger Games",
        case=False,
        na=False
    )
][["book_id", "title"]]

,book_id,title
0,1,"The Hunger Games (The Hunger Games, #1)"
16,17,"Catching Fire (The Hunger Games, #2)"
19,20,"Mockingjay (The Hunger Games, #3)"
506,507,The Hunger Games Trilogy Boxset (The Hunger Ga...
716,717,The Hunger Games: Official Illustrated Movie C...
1354,1355,The Hunger Games Tribute Guide
6223,6224,The World of the Hunger Games (Hunger Games Tr...
8576,8577,The Girl Who Was on Fire: Your Favorite Author...


In [97]:
print("cosine_sim" in globals())

False


In [98]:
print([x for x in globals() if "cosine" in x.lower()])

['cosine_similarity']


In [99]:
type(cosine_similarity)

function

In [102]:
recommend_content(
    "The Hunger Games (The Hunger Games, #1)"
)

,book_id,title,authors,similarity
0,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.890083
1,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.862205
2,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.763699
3,12,"Divergent (Divergent, #1)",Veronica Roth,0.700538
4,91,"The Maze Runner (Maze Runner, #1)",James Dashner,0.617344
5,327,"Legend (Legend, #1)",Marie Lu,0.595743
6,69,"Insurgent (Divergent, #2)",Veronica Roth,0.589065
7,2301,"The Ask and the Answer (Chaos Walking, #2)",Patrick Ness,0.578314
8,525,"Pandemonium (Delirium, #2)",Lauren Oliver,0.563836
9,655,"Crossed (Matched, #2)",Ally Condie,0.560962
